In [1]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

# Fetch Credit Approval dataset from UCI
credit_data = fetch_ucirepo(id=27)

# Separate features and target
X = credit_data.data.features
y = credit_data.data.targets

# Combine into a single DataFrame
df = pd.concat([X, y], axis=1)

In [2]:
# Dataset overview
print("Dataset shape:", df.shape)

print("\nFeature names:")
print(X.columns.tolist())

print("\nTarget column:")
print(y.columns.tolist())

print("\nFirst five observations:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())

print("\nTarget distribution:")
print(y.value_counts())

# Display summary statistics
print("\nDataset summary:")
display(X.describe(include="all"))

Dataset shape: (690, 16)

Feature names:
['A15', 'A14', 'A13', 'A12', 'A11', 'A10', 'A9', 'A8', 'A7', 'A6', 'A5', 'A4', 'A3', 'A2', 'A1']

Target column:
['A16']

First five observations:


,A15,A14,A13,A12,A11,A10,A9,A8,A7,A6,A5,A4,A3,A2,A1,A16
0,0,202.0,g,f,1,t,t,1.25,v,w,g,u,0.000,30.83,b,+
1,560,43.0,g,f,6,t,t,3.04,h,q,g,u,4.460,58.67,a,+
2,824,280.0,g,f,0,f,t,1.50,h,q,g,u,0.500,24.50,a,+
3,3,100.0,g,t,5,t,t,3.75,v,w,g,u,1.540,27.83,b,+
4,0,120.0,s,f,0,f,t,1.71,v,w,g,u,5.625,20.17,b,+



Missing values:
A15     0
A14    13
A13     0
A12     0
A11     0
A10     0
A9      0
A8      0
A7      9
A6      9
A5      6
A4      6
A3      0
A2     12
A1     12
A16     0
dtype: int64

Target distribution:
A16
-      383
+      307
Name: count, dtype: int64

Dataset summary:


,A15,A14,A13,A12,A11,A10,A9,A8,A7,A6,A5,A4,A3,A2,A1
count,690.000000,677.000000,690,690,690.00000,690,690,690.000000,681,681,684,684,690.000000,678.000000,678
unique,NaN,NaN,3,2,NaN,2,2,NaN,9,14,3,3,NaN,NaN,2
top,NaN,NaN,g,f,NaN,f,t,NaN,v,c,g,u,NaN,NaN,b
freq,NaN,NaN,625,374,NaN,395,361,NaN,399,137,519,519,NaN,NaN,468
mean,1017.385507,184.014771,NaN,NaN,2.40000,NaN,NaN,2.223406,NaN,NaN,NaN,NaN,4.758725,31.568171,NaN
std,5210.102598,173.806768,NaN,NaN,4.86294,NaN,NaN,3.346513,NaN,NaN,NaN,NaN,4.978163,11.957862,NaN
min,0.000000,0.000000,NaN,NaN,0.00000,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,0.000000,13.750000,NaN
25%,0.000000,75.000000,NaN,NaN,0.00000,NaN,NaN,0.165000,NaN,NaN,NaN,NaN,1.000000,22.602500,NaN
50%,5.000000,160.000000,NaN,NaN,0.00000,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,2.750000,28.460000,NaN
75%,395.500000,276.000000,NaN,NaN,3.00000,NaN,NaN,2.625000,NaN,NaN,NaN,NaN,7.207500,38.230000,NaN


In [3]:
# ============================================================
# DATA PREPROCESSING AND TRAIN-TEST SPLIT
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Define numerical and categorical columns
numerical_features = [
    "A2", "A3", "A8", "A11", "A14", "A15"
]

categorical_features = [
    "A1", "A4", "A5", "A6", "A7",
    "A9", "A10", "A12", "A13"
]

# Prepare independent variables
X = df.drop(columns=["A16"]).copy()

# Convert target into binary values
# + represents approval and - represents rejection
y = df["A16"].map({"+": 1, "-": 0})

# Verify target conversion
assert y.notna().all(), "Unexpected target values found"

# Numerical preprocessing
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

# Combine preprocessing operations
preprocessor = ColumnTransformer(transformers=[
    ("numerical", numerical_transformer, numerical_features),
    ("categorical", categorical_transformer, categorical_features)
])

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Display split information
print("Training observations:", X_train.shape[0])
print("Testing observations:", X_test.shape[0])

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

print("\nPreprocessing configuration completed.")

Training observations: 552
Testing observations: 138

Training target distribution:
A16
0    306
1    246
Name: count, dtype: int64

Testing target distribution:
A16
0    77
1    61
Name: count, dtype: int64

Preprocessing configuration completed.


In [4]:
# Import evaluation and storage utilities

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

# Dictionary to store the trained models
trained_models = {}

# List to store evaluation results
evaluation_results = []

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Initialize Logistic Regression
logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

# Train the model
logistic_model.fit(X_train, y_train)

# Save the trained model
trained_models["Logistic Regression"] = logistic_model

print("Logistic Regression training completed.")

Logistic Regression training completed.


In [6]:
# Predict test observations
lr_predictions = logistic_model.predict(X_test)

# Calculate prediction probabilities
lr_probabilities = logistic_model.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
lr_metrics = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, lr_predictions),
    "AUC": roc_auc_score(y_test, lr_probabilities),
    "Precision": precision_score(y_test, lr_predictions, zero_division=0),
    "Recall": recall_score(y_test, lr_predictions, zero_division=0),
    "F1 Score": f1_score(y_test, lr_predictions, zero_division=0),
    "MCC": matthews_corrcoef(y_test, lr_predictions)
}

evaluation_results.append(lr_metrics)

print("Logistic Regression Results:")
display(pd.DataFrame([lr_metrics]).round(4))

Logistic Regression Results:


,Model,Accuracy,AUC,Precision,Recall,F1 Score,MCC
0,Logistic Regression,0.8696,0.9602,0.8525,0.8525,0.8525,0.7356


In [7]:
from sklearn.tree import DecisionTreeClassifier

# Initialize Decision Tree
decision_tree_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        max_depth=6,
        random_state=42
    ))
])

# Train the model
decision_tree_model.fit(X_train, y_train)

# Save the trained model
trained_models["Decision Tree"] = decision_tree_model

print("Decision Tree training completed.")

Decision Tree training completed.


In [8]:
dt_predictions = decision_tree_model.predict(X_test)
dt_probabilities = decision_tree_model.predict_proba(X_test)[:, 1]

dt_metrics = {
    "Model": "Decision Tree",
    "Accuracy": accuracy_score(y_test, dt_predictions),
    "AUC": roc_auc_score(y_test, dt_probabilities),
    "Precision": precision_score(y_test, dt_predictions, zero_division=0),
    "Recall": recall_score(y_test, dt_predictions, zero_division=0),
    "F1 Score": f1_score(y_test, dt_predictions, zero_division=0),
    "MCC": matthews_corrcoef(y_test, dt_predictions)
}

evaluation_results.append(dt_metrics)

print("Decision Tree Results:")
display(pd.DataFrame([dt_metrics]).round(4))

Decision Tree Results:


,Model,Accuracy,AUC,Precision,Recall,F1 Score,MCC
0,Decision Tree,0.8913,0.934,0.8382,0.9344,0.8837,0.7863


In [9]:
from sklearn.neighbors import KNeighborsClassifier

# Initialize KNN
knn_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", KNeighborsClassifier(n_neighbors=5))
])

# Train the model
knn_model.fit(X_train, y_train)

# Save the trained model
trained_models["KNN"] = knn_model

print("KNN training completed.")

KNN training completed.


In [10]:
knn_predictions = knn_model.predict(X_test)
knn_probabilities = knn_model.predict_proba(X_test)[:, 1]

knn_metrics = {
    "Model": "KNN",
    "Accuracy": accuracy_score(y_test, knn_predictions),
    "AUC": roc_auc_score(y_test, knn_probabilities),
    "Precision": precision_score(y_test, knn_predictions, zero_division=0),
    "Recall": recall_score(y_test, knn_predictions, zero_division=0),
    "F1 Score": f1_score(y_test, knn_predictions, zero_division=0),
    "MCC": matthews_corrcoef(y_test, knn_predictions)
}

evaluation_results.append(knn_metrics)

print("KNN Results:")
display(pd.DataFrame([knn_metrics]).round(4))

KNN Results:


,Model,Accuracy,AUC,Precision,Recall,F1 Score,MCC
0,KNN,0.8623,0.9267,0.875,0.8033,0.8376,0.7205


In [11]:
from sklearn.naive_bayes import GaussianNB

# Initialize Gaussian Naive Bayes
naive_bayes_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", GaussianNB())
])

# Train the model
naive_bayes_model.fit(X_train, y_train)

# Save the trained model
trained_models["Naive Bayes"] = naive_bayes_model

print("Gaussian Naive Bayes training completed.")

Gaussian Naive Bayes training completed.


In [12]:
nb_predictions = naive_bayes_model.predict(X_test)
nb_probabilities = naive_bayes_model.predict_proba(X_test)[:, 1]

nb_metrics = {
    "Model": "Naive Bayes",
    "Accuracy": accuracy_score(y_test, nb_predictions),
    "AUC": roc_auc_score(y_test, nb_probabilities),
    "Precision": precision_score(y_test, nb_predictions, zero_division=0),
    "Recall": recall_score(y_test, nb_predictions, zero_division=0),
    "F1 Score": f1_score(y_test, nb_predictions, zero_division=0),
    "MCC": matthews_corrcoef(y_test, nb_predictions)
}

evaluation_results.append(nb_metrics)

print("Naive Bayes Results:")
display(pd.DataFrame([nb_metrics]).round(4))

Naive Bayes Results:


,Model,Accuracy,AUC,Precision,Recall,F1 Score,MCC
0,Naive Bayes,0.7464,0.903,0.9333,0.459,0.6154,0.5214


In [13]:
# Importing the required classes for Support Vector Machine
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline

# SVM is useful for finding a suitable boundary between
# different classes of data.
# Here, an RBF kernel is used to handle possible non-linear patterns.

# Probability estimates are required for calculating AUC.
# CalibratedClassifierCV is used to obtain these probabilities
# without using the deprecated probability parameter of SVC.

svm_estimator = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", SVC(
        kernel="rbf",
        random_state=42
    ))
])

svm_model = CalibratedClassifierCV(
    estimator=svm_estimator,
    method="sigmoid",
    cv=5,
    ensemble=False
)

# Fit the model using only the training observations.
# The testing observations are kept separate for evaluation.
svm_model.fit(X_train, y_train)

# Store the trained model for later use in the application.
trained_models["Support Vector Machine"] = svm_model

print("Support Vector Machine training completed.")

Support Vector Machine training completed.


In [14]:
# Predict the class labels for the test observations.
svm_predictions = svm_model.predict(X_test)

# Get the estimated probability of belonging to class 1.
# These probability values are used to calculate the AUC score.
svm_probabilities = svm_model.predict_proba(X_test)[:, 1]

# Calculate the six performance measures required for the assignment.
svm_metrics = {
    "Model": "Support Vector Machine",

    # Percentage of correctly classified observations
    "Accuracy": accuracy_score(y_test, svm_predictions),

    # Measures how well the model separates the two classes
    "AUC": roc_auc_score(y_test, svm_probabilities),

    # Out of all positive predictions, how many were correct
    "Precision": precision_score(
        y_test, svm_predictions, zero_division=0
    ),

    # Out of all actual positive observations, how many were identified
    "Recall": recall_score(
        y_test, svm_predictions, zero_division=0
    ),

    # Harmonic mean of precision and recall
    "F1 Score": f1_score(
        y_test, svm_predictions, zero_division=0
    ),

    # Balanced measure of classification performance
    "MCC": matthews_corrcoef(y_test, svm_predictions)
}

# Add the SVM results to the overall comparison
evaluation_results.append(svm_metrics)

print("Support Vector Machine evaluation results:")
display(pd.DataFrame([svm_metrics]).round(4))

Support Vector Machine evaluation results:


,Model,Accuracy,AUC,Precision,Recall,F1 Score,MCC
0,Support Vector Machine,0.8768,0.9557,0.8438,0.8852,0.864,0.7523


In [15]:
# Combine all model evaluation results
results_df = pd.DataFrame(evaluation_results)

# Display comparison table
display(results_df.round(4))

,Model,Accuracy,AUC,Precision,Recall,F1 Score,MCC
0,Logistic Regression,0.8696,0.9602,0.8525,0.8525,0.8525,0.7356
1,Decision Tree,0.8913,0.9340,0.8382,0.9344,0.8837,0.7863
2,KNN,0.8623,0.9267,0.8750,0.8033,0.8376,0.7205
3,Naive Bayes,0.7464,0.9030,0.9333,0.4590,0.6154,0.5214
4,Support Vector Machine,0.8768,0.9557,0.8438,0.8852,0.8640,0.7523


In [16]:
# ============================================================
# FINAL STEP: SAVE MODELS AND EXPERIMENTAL RESULTS
# ============================================================

from pathlib import Path
import joblib

# Set the main assignment folder.
# All output files will be stored inside this location.
PROJECT_DIR = Path(
    r"C:\Users\samudrala.vinay\AIMLZG565_Assignment2"
)

# Create the required folders if they are not available.
MODEL_DIR = PROJECT_DIR / "models"
RESULT_DIR = PROJECT_DIR / "results"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# Save all five trained classification pipelines.
# Each saved pipeline includes its preprocessing steps
# and the corresponding trained classification algorithm.

for model_name, model in trained_models.items():

    # Prepare a file name using the model name.
    file_name = model_name.lower().replace(" ", "_") + ".joblib"

    # Define where the model should be stored.
    model_path = MODEL_DIR / file_name

    # Save the trained pipeline.
    joblib.dump(model, model_path)

    print(f"{model_name} saved successfully.")

# Store the original test observations along with their actual labels.
# This CSV will be uploaded to the Streamlit application for evaluation.

test_data = X_test.copy()
test_data["A16"] = y_test

test_data.to_csv(PROJECT_DIR / "test_data.csv", index=False)

print("\nTest dataset saved successfully.")

# Save the comparison of all five classifiers.
# These results will be used in the README and final report.

results_df.to_csv(
    RESULT_DIR / "model_comparison.csv",
    index=False
)

print("Model comparison results saved successfully.")

print("\nAll assignment outputs have been saved.")

Logistic Regression saved successfully.
Decision Tree saved successfully.
KNN saved successfully.
Naive Bayes saved successfully.
Support Vector Machine saved successfully.

Test dataset saved successfully.
Model comparison results saved successfully.

All assignment outputs have been saved.
